<a href="https://colab.research.google.com/github/ArinaKrasotina/data-analysis/blob/main/%D0%9A%D1%80%D0%B0%D1%81%D0%BE%D1%82%D0%B8%D0%BD%D0%B0_%22%D0%9A%D0%B5%D0%B9%D1%81_%D0%9B%D0%BE%D0%B3%D0%B8%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1 кейс**

**Работа с логами**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [ ]:
!wget https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log

--2025-11-26 07:41:29--  https://gist.github.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.github.com (gist.github.com)... 140.82.121.3
Connecting to gist.github.com (gist.github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log [following]
--2025-11-26 07:41:29--  https://gist.githubusercontent.com/Vs8th/38d5d914171c84166728a9746d212bad/raw/auto_purchase.log
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 459418 (449K) [text/plain]
Saving to: ‘auto_purchase.log’

auto_purchase.log   100%[===================>] 448.65K  --.-KB/s    in 0.02s   

2025-11-26 07:41:30 (27.8

Чтобы посмотреть как он выглядит выполните следующую ячейку.

In [ ]:
with open('auto_purchase.log', 'r') as f:
    lines = f.readlines()

for line in lines[400:550]:
    print(line)

INFO  | 2022-11-26 00:01:03,024  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-26, количество людей с автопродлением подписки: 0

INFO  | 2022-11-26 00:01:03,027  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы

INFO  | 2022-11-27 00:01:02,455  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-27 00:01:02,480  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-27, количество людей с автопродлением подписки: 0

INFO  | 2022-11-27 00:01:02,483  | file: demon_auto_purchase_bundle.py |  line: 57 | [demon] Выход из программы

INFO  | 2022-11-28 00:01:02,406  | file: demon_auto_purchase_bundle.py |  line: 44 | [demon] Поверяем истекающие премиум-пакеты

INFO  | 2022-11-28 00:01:02,424  | file: demon_auto_purchase_bundle.py |  line: 54 | [demon] Cегодня 2022-11-28, количество людей с автопродлением подписки: 0

INFO  | 2022-11-28 00:01:02,427  | f

### **Решения**

#### **Задача 1**

Ваша задача написать функцию `count_success_and_failure`, которая принимает на вход путь к файлу с логами и подсчитывает количество успешных продлений и ошибок при списании. Функция должна вернуть кортеж из двух значений: количества успешных попыток и неуспешных.

**Решение**

Напишите свое решение ниже

In [ ]:
def count_success_and_failure(file_path):
    total_updates = 0
    error_count = 0

    with open(file_path, 'r') as f:
        for line in f:
            if 'Обновляем подписку пользователю' in line:
                total_updates += 1
            if 'ошибка при списании' in line:
                error_count += 1

    success_count = total_updates - error_count
    failure_count = error_count

    return (success_count, failure_count)


count_success_and_failure('auto_purchase.log')

(1034, 186)

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
res = count_success_and_failure('auto_purchase.log')

try:
    assert res == (1034, 186)
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача написать функцию `auto_renewal_sub`, которая принимает на вход путь к файлу с логами и обрабатывает количество клиентов с автопродлением подписки. Мы хотим посмотреть на изменение этого показателя в динамике: посчитайте сглаженные значения с помощью метода скользящего среднего и метода медианного сглаживания.  

**Примечание:** При сглаживании берем все предыдущие значения, включая текущее, будущие значения не берем. Если в один день наблюдаем несколько записей об автопродлении - берем максимальное из имеющихся число клиентов с подпиской.

Функция должна записать в файл `auto_renewal_sub.txt` два списка, предварив их соответствущими обозначениями:

`Среднее: [2.0, 1.0, 0.67...]`

`Медиана: [2, 2, 0...]`

**Решение**

Напишите свое решение ниже

In [ ]:
import statistics
from datetime import datetime
import re

def auto_renewal_sub(log_file_path):
    daily_counts = {}

    with open(log_file_path, 'r') as f:
        for line in f:
            if 'количество людей с автопродлением подписки:' in line:
                try:
                    date_match = re.search(r'\d{4}-\d{2}-\d{2}', line)
                    if date_match:
                        date_str = date_match.group(0)

                        count_match = re.search(r'количество людей с автопродлением подписки:\s*(\d+)', line)
                        if count_match:
                            count = int(count_match.group(1))

                            if date_str not in daily_counts or count > daily_counts[date_str]:
                                daily_counts[date_str] = count
                except:
                    continue

    if not daily_counts:
        with open('auto_renewal_sub.txt', 'w') as f:
            f.write("Среднее: []\n")
            f.write("Медиана: []\n")
        return [], []

    sorted_dates = sorted(daily_counts.keys())
    values = [daily_counts[date] for date in sorted_dates]


    moving_avg = []
    moving_median = []

    for i in range(len(values)):

        window = values[:i+1]


        avg = sum(window) / len(window)
        moving_avg.append(round(avg, 2))


        median = statistics.median(window)
        moving_median.append(int(median))


    with open('auto_renewal_sub.txt', 'w') as f:
        f.write(f"Среднее: {moving_avg}\n")
        f.write(f"Медиана: {moving_median}\n")

    return moving_avg, moving_median


auto_renewal_sub('auto_purchase.log')

([1.0,
  0.5,
  0.33,
  0.25,
  0.2,
  0.17,
  0.14,
  0.12,
  0.11,
  0.1,
  0.09,
  0.08,
  0.08,
  0.07,
  0.07,
  0.06,
  0.06,
  0.06,
  0.05,
  0.05,
  0.05,
  0.05,
  0.04,
  0.04,
  0.04,
  0.04,
  0.04,
  0.04,
  0.03,
  0.03,
  0.06,
  0.09,
  0.09,
  0.09,
  0.09,
  0.08,
  0.08,
  0.08,
  0.08,
  0.07,
  0.07,
  0.07,
  0.12,
  0.14,
  0.13,
  0.13,
  0.13,
  0.15,
  0.16,
  0.16,
  0.16,
  0.15,
  0.15,
  0.15,
  0.15,
  0.14,
  0.16,
  0.17,
  0.17,
  0.17,
  0.2,
  0.21,
  0.21,
  0.2,
  0.22,
  0.21,
  0.21,
  0.21,
  0.2,
  0.2,
  0.2,
  0.19,
  0.21,
  0.2,
  0.2,
  0.2,
  0.19,
  0.21,
  0.22,
  0.21,
  0.22,
  0.22,
  0.22,
  0.21,
  0.21,
  0.21,
  0.22,
  0.22,
  0.21,
  0.22,
  0.24,
  0.24,
  0.25,
  0.24,
  0.24,
  0.25,
  0.25,
  0.24,
  0.24,
  0.25,
  0.25,
  0.25,
  0.24,
  0.24,
  0.24,
  0.24,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.23,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.24,
  0.

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [ ]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt

import pandas as pd

user_answer = pd.read_csv('auto_renewal_sub.txt')
correct_answer = pd.read_csv('cor_auto_renewal.txt')

--2025-11-26 08:01:44--  https://gist.github.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.github.com (gist.github.com)... 140.82.121.4
Connecting to gist.github.com (gist.github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt [following]
--2025-11-26 08:01:44--  https://gist.githubusercontent.com/Vs8th/846cbc97a722ab825cda4dda3b2b3434/raw/cor_auto_renewal.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2431 (2.4K) [text/plain]
Saving to: ‘cor_auto_renewal.txt’

cor_auto_renewal.tx 100%[===================>]   2.37K  --.-KB/s    in 0s      

2025-11-26 08:0

In [ ]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!


#### **Задача 3**

Напишите функцию `sub_renewal_by_day`, которая принимает на вход путь к файлу с логами и анализирует взаимосвязь дня продления подписки и количества продлений в этот день. Функция должна записать в файл `weekdays.txt` аналитическую записку в формате:

**`Количество обновлений подписки по дням недели:`**

**`Понедельник: 6`**

**`Вторник: 7`**

**`Среда: 8`**

**`...`**

**Решение**

Напишите свое решение ниже

In [ ]:
import datetime

def sub_renewal_by_day(log_path):
    renewals_by_day = {"Понедельник": 0, "Вторник": 0, "Среда": 0, "Четверг": 0, "Пятница": 0, "Суббота": 0, "Воскресенье": 0}
    weekdays = ["Понедельник", "Вторник", "Среда", "Четверг", "Пятница", "Суббота", "Воскресенье"]

    renewal_status = {}
    all_renewals = []

    with open(log_path, "r", encoding="utf-8") as file:
        lines = file.readlines()

    for i, line in enumerate(lines):
        if "Обновляем подписку пользователю id:" in line:
            parts = line.split("|")
            date_str = parts[1].strip().split(",")[0]
            date = datetime.datetime.strptime(date_str, "%Y-%m-%d %H:%M:%S")

            start_index = line.find("id:") + 3
            end_index = line.find(", payment_method_id")
            user_id = line[start_index:end_index].strip()

            renewal_key = (user_id, date)
            renewal_status[renewal_key] = True

            all_renewals.append({
                'key': renewal_key,
                'date': date,
                'user_id': user_id,
                'line_num': i
            })

    for i, line in enumerate(lines):
        if "ERROR" in line and "ошибка при списании" in line:
            start_index = line.find("id:") + 3
            user_id_from_error = line[start_index:].strip().split()[0]

            closest_renewal = None
            closest_distance = float('inf')

            for renewal in all_renewals:
                if renewal['user_id'] == user_id_from_error and renewal['line_num'] < i:
                    distance = i - renewal['line_num']
                    if distance < closest_distance:
                        closest_distance = distance
                        closest_renewal = renewal

            if closest_renewal and closest_distance <= 50:
                renewal_status[closest_renewal['key']] = False

    for renewal in all_renewals:
        if renewal_status[renewal['key']]:
            weekday_name = weekdays[renewal['date'].weekday()]
            renewals_by_day[weekday_name] += 1

    with open("weekdays.txt", "w", encoding="utf-8") as f_out:
        f_out.write("Количество обновлений подписки по дням недели:\n")
        for day in weekdays:
            f_out.write(f"{day}: {renewals_by_day[day]}\n")

    return renewals_by_day

sub_renewal_by_day('auto_purchase.log')

{'Понедельник': 136,
 'Вторник': 144,
 'Среда': 162,
 'Четверг': 169,
 'Пятница': 145,
 'Суббота': 135,
 'Воскресенье': 143}

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [59]:
# Здесь будет скачиваться файл с эталонным ответом

!wget https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt

import pandas as pd

user_answer = pd.read_csv('weekdays.txt')
correct_answer = pd.read_csv('cor_weekdays.txt')

--2025-11-26 09:45:38--  https://gist.github.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.github.com (gist.github.com)... 140.82.121.3
Connecting to gist.github.com (gist.github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt [following]
--2025-11-26 09:45:38--  https://gist.githubusercontent.com/Vs8th/c4c382f50761b5b9e64795eb89d49fda/raw/cor_weekdays.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 238 [text/plain]
Saving to: ‘cor_weekdays.txt.1’

cor_weekdays.txt.1  100%[===================>]     238  --.-KB/s    in 0s      

2025-11-26 09:45:39 (4.18 MB/s) - ‘co

In [60]:
try:
  assert (user_answer == correct_answer).all().all(), 'Ответы не совпадают'
except Exception as err:
  raise AssertionError(f'При проверке возникла ошибка {repr(err)}')
else:
  print('Поздравляем, Вы справились и успешно прошли все проверки!!')

Поздравляем, Вы справились и успешно прошли все проверки!!
